# 读取文件

In [ ]:
import pandas as pd

df = pd.read_csv("../data/6.匹配地点/address_transport_result_4_4.csv")
print(df.head())

# 格式化时间

In [ ]:
# 把原始 case_time 列统一转换成 pandas 的字符串类型（StringDtype），并去掉首尾空格
# 这样后面用正则提取时更稳健（避免因为空格导致匹配失败）
s = df["case_time"].astype("string").str.strip()

# 定义用于提取“年/月/日/时/分”的正则表达式（中文日期格式）
# 这里用“命名捕获组” (?P<name>...)，提取后会直接得到列名 year/month/day/hour/minute
pat = (
    r"(?P<year>\d{4})年"          # year: 4位数字，例如 2021年
    r"(?P<month>\d{1,2})月"       # month: 1~2位数字，例如 8月 或 08月
    r"(?P<day>\d{1,2})日"         # day: 1~2位数字，例如 16日
    r"(?P<hour>\d{1,2})时"        # hour: 1~2位数字，例如 11时
    r"(?:(?P<minute>\d{1,2})分)?" # minute: 可选（? 表示出现0次或1次）
                                 # 外层 (?:...) 是“非捕获组”，避免产生多余列
                                 # 如果没有“xx分”，minute 会是缺失值 NA
)

# 对 s 逐行应用正则提取：
# - 能匹配到的行：返回 year/month/day/hour/minute 五列（minute 可能为空）
# - 不能匹配的行（例如：只有“2021年”、未知、空）：对应列都是 NA
parts = s.str.extract(pat)

# 只保留“精确到小时”的记录：
# 要求 year/month/day/hour 这四个字段都不为空（notna）
# all(axis=1) 表示按行判断：这一行四列都非空才为 True
keep = parts[["year", "month", "day", "hour"]].notna().all(axis=1)

# 用 keep 过滤原始数据 df，得到符合条件的新表 df2
# copy() 复制一份，避免后续修改触发 SettingWithCopyWarning
df2 = df.loc[keep].copy()

# 同样过滤提取结果 parts，得到 parts2（与 df2 行一一对应）
parts2 = parts.loc[keep].copy()

# str.extract 得到的是字符串列（StringDtype），但 to_datetime 需要数值
# 所以把 year/month/day/hour/minute 统一转成数值：
# errors="coerce" 表示：遇到无法转换的内容就变成 NaN（而不是报错）
for c in ["year", "month", "day", "hour", "minute"]:
    parts2[c] = pd.to_numeric(parts2[c], errors="coerce")

#  [新增这一步] 丢弃掉年月日时中因为解析错误而变成 NaN 的行
parts2 = parts2.dropna(subset=["year", "month", "day", "hour"]).copy()

# minute 是可选字段：
# - 如果原字符串没有“xx分”，minute 为 NaN
# - 我们希望保留这些记录，并把 minute 当成 0 分钟（整点）
# 因此先 fillna(0)，再转成 int64
parts2["minute"] = parts2["minute"].fillna(0).astype("int64")

# 生成 pandas datetime：
# pd.to_datetime 可以接收一个 dict（year/month/day/hour/minute），逐行组成时间
# errors="coerce" 表示：如果日期非法（比如 2021年13月40日），会变成 NaT 而不是报错
dt = pd.to_datetime(
    dict(
        year=parts2["year"].astype("int64"),   # 年转 int
        month=parts2["month"].astype("int64"), # 月转 int
        day=parts2["day"].astype("int64"),     # 日转 int
        hour=parts2["hour"].astype("int64"),   # 时转 int
        minute=parts2["minute"],               # 分已是 int64
    ),
    errors="coerce",
)

# 把生成好的 datetime 写回 df2
df2["case_dt"] = dt

# 再保险过滤一次：把无法生成 datetime 的行去掉（NaT）
df2 = df2[df2["case_dt"].notna()].copy()

# 截断到“小时”粒度：
# - 例如 2020-11-10 08:40 -> 2020-11-10 08:00
# - 原本就是整点（minute=0）的不会变
# 方便做小时分布统计（0-23点、或按时间序列聚合）
df2["case_dt_hour"] = df2["case_dt"].dt.floor("H")


In [ ]:
df2

# 导出检查数据

In [ ]:
# 筛选列 进行保存
cols = ["案号", "裁判日期", "当事人", 
        "province", "city", "district", "specific_place",
        "vehicle", "death",
        "lng", "lat", 
        "case_dt_hour"]

df2[cols].to_csv("../data/7.匹配时间/time_transport_result_4_4.csv",index=False,encoding="utf-8-sig")
